## Importing necessary packages

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, callbacks
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


## Setting up dataset paths and output directories

In [ ]:
# Celeb-DF v2
CELEB_REAL_DIR = "//kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-real"
CELEB_FAKE_DIR = "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis"

# FaceForensics++ C23
FF_BASE = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"

FF_REAL_DIR = f"{FF_BASE}/original"
FF_DEEPFAKES_DIR = f"{FF_BASE}/Deepfakes"
FF_FACE2FACE_DIR = f"{FF_BASE}/Face2Face"
FF_FACESWAP_DIR = f"{FF_BASE}/FaceSwap"
FF_FACESHIFTER_DIR = f"{FF_BASE}/FaceShifter"
FF_NEURALTEXTURES_DIR = f"{FF_BASE}/NeuralTextures"
FF_DFD_DIR = f"{FF_BASE}/DeepFakeDetection"

BASE_OUTPUT = "/kaggle/working/frames"
REAL_DIR = os.path.join(BASE_OUTPUT, "real")
FAKE_DIR = os.path.join(BASE_OUTPUT, "fake")

os.makedirs(REAL_DIR, exist_ok=True)
os.makedirs(FAKE_DIR, exist_ok=True)

print("Folders created successfully")

## Face Extraction Function using Haar Cascades

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)


def extract_face(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5
    )

    if len(faces) == 0:
        return None

    largest_face = max(faces, key=lambda x: x[2] * x[3])
    x, y, w, h = largest_face

    face = frame[y:y+h, x:x+w]

    if face.shape[0] < 50 or face.shape[1] < 50:
        return None

    return face

## Function to get all video paths from a directory (with a limit)

In [ ]:
def get_all_video_paths(root_dir, max_videos=500):
    video_paths = []

    for root, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith((".mp4", ".avi", ".mov", ".mkv")):
                video_paths.append(os.path.join(root, file))

    return video_paths[:max_videos]



def extract_frames_from_videos(videos_dir, output_dir, label, max_videos=500):
    if not os.path.exists(videos_dir):
        print(f"Path not found: {videos_dir}")
        return

    video_files = get_all_video_paths(videos_dir, max_videos=max_videos)

    print(f"Processing {len(video_files)} videos from {videos_dir}")

    total_saved = 0

    for video_path in video_files:
        video_name = os.path.basename(video_path)
        cap = cv2.VideoCapture(video_path)

        fps = int(cap.get(cv2.CAP_PROP_FPS))
        if fps == 0:
            fps = 30

        # 2 FPS extraction
        frame_interval = max(1, fps // 2)

        frame_count = 0
        saved_count = 0

        success, frame = cap.read()

        while success:
            if frame_count % frame_interval == 0:
                face = extract_face(frame)

                if face is not None:
                    face = cv2.resize(face, (224, 224))

                    filename = f"{label}_{video_name}_frame_{saved_count}.jpg"
                    save_path = os.path.join(output_dir, filename)

                    cv2.imwrite(save_path, face)
                    saved_count += 1
                    total_saved += 1

            success, frame = cap.read()
            frame_count += 1

        cap.release()

    print(f"Saved {total_saved} frames to {output_dir}")

## Extracting frames from REAL and FAKE videos with specified limits

In [ ]:
print("Extracting REAL videos...")
extract_frames_from_videos(FF_REAL_DIR, REAL_DIR, "ff_real", max_videos=500)
extract_frames_from_videos(CELEB_REAL_DIR, REAL_DIR, "celeb_real", max_videos=500)

fake_sources = [
    FF_DEEPFAKES_DIR,
    FF_FACE2FACE_DIR,
    FF_FACESWAP_DIR,
    FF_FACESHIFTER_DIR,
    FF_NEURALTEXTURES_DIR,
    FF_DFD_DIR
]

for idx, source in enumerate(fake_sources):
    print(f"Processing fake source {idx + 1}: {source}")
    extract_frames_from_videos(
        source,
        FAKE_DIR,
        f"fake_{idx}",
        max_videos=150
    )

extract_frames_from_videos(CELEB_FAKE_DIR, FAKE_DIR, "celeb_fake", max_videos=300)

print("Extraction completed")
print("REAL frames:", len(os.listdir(REAL_DIR)))
print("FAKE frames:", len(os.listdir(FAKE_DIR)))

## Data agumentation and preparing the data for model training

In [ ]:
dataset_dir = BASE_OUTPUT
batch_size = 32
img_size = (224, 224)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
])

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

print("Class Names:", train_ds.class_names)

val_batches = tf.data.experimental.cardinality(val_test_ds)
val_ds = val_test_ds.take(val_batches // 2)
test_ds = val_test_ds.skip(val_batches // 2)


def prepare(ds, augment=False):
    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    ds = ds.map(
        lambda x, y: (tf.cast(x, tf.float32), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = prepare(train_ds, augment=True)
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)

## Created the base model before training

In [ ]:
base_model = tf.keras.applications.EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
    pooling="avg"
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)

x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(2, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Setting up callbacks and starting with the first phase of training

In [ ]:
callbacks_list = [
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        "./sentinel_video_model.keras",
        monitor="val_accuracy",
        save_best_only=True
    )
]

print("Starting Phase 1 Training...")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_list
)

## Fine tuning the model when required

In [ ]:
print("Starting Fine-Tuning...")

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

print("Trainable layers:")
for layer in base_model.layers[-10:]:
    print(layer.name, layer.trainable)

model.compile( optimizer=tf.keras.optimizers.AdamW(
    learning_rate=1e-5,
    weight_decay=1e-5
), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks_list)

## For the testing the model later in backend

In [ ]:
model.load_weights("./sentinel_video_model.keras")


def predict_video(video_path, threshold=0.4):
    cap = cv2.VideoCapture(video_path)

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps == 0:
        fps = 30

    frame_interval = max(1, fps // 2)

    frame_count = 0
    fake_count = 0
    total_count = 0

    success, frame = cap.read()

    while success:
        if frame_count % frame_interval == 0:
            face = extract_face(frame)

            if face is not None:
                face = cv2.resize(face, (224, 224))
                face = np.expand_dims(face, axis=0)
                face = face.astype(np.float32)

                pred = model.predict(face, verbose=0)
                pred_class = np.argmax(pred)

                # Verify using print(train_ds.class_names)
                if pred_class == 0:
                    fake_count += 1

                total_count += 1

        success, frame = cap.read()
        frame_count += 1

    cap.release()

    if total_count == 0:
        return "Unable to detect"

    fake_ratio = fake_count / total_count

    if fake_ratio > threshold:
        return f"FAKE ({fake_ratio:.2%})"
    else:
        return f"REAL ({fake_ratio:.2%})"

print("Notebook ready for training + inference")


## Evaluating the model on test dataset

In [ ]:
model.load_weights("./sentinel_video_model.keras")

print("Evaluating model on test dataset...\n")

test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

## Set up the classification report for the model

In [ ]:
all_labels = []
all_predictions = []
all_probabilities = []

for images, labels in test_ds:

    preds = model.predict(images, verbose=0)

    predicted_classes = np.argmax(preds, axis=1)

    all_predictions.extend(predicted_classes)
    all_labels.extend(labels.numpy())

    # probability of fake class (index 0 because class_names = ['fake', 'real'])
    all_probabilities.extend(preds[:, 0])

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

In [ ]:
print("\nTrue label distribution:")
unique_true, counts_true = np.unique(all_labels, return_counts=True)
print(dict(zip(unique_true, counts_true)))

print("\nPrediction distribution:")
unique_pred, counts_pred = np.unique(all_predictions, return_counts=True)
print(dict(zip(unique_pred, counts_pred)))

print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================\n")
print( classification_report( all_labels, all_predictions, target_names=["fake", "real"], zero_division=0 ) )

In [ ]:
final_accuracy = accuracy_score(all_labels, all_predictions)
print(f"\nFinal Accuracy Score: {final_accuracy * 100:.2f}%")

cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap( cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Pred Fake", "Pred Real"], yticklabels=["Actual Fake", "Actual Real"] )
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nHOW TO READ RESULTS:")
print("- High recall for FAKE class is VERY important")
print("- Missing fake videos is worse than false positives")
print("- Precision + Recall matter more than only accuracy")
print("- Confusion matrix should not be heavily biased")